# Presentation of our solution -- Quantum urban vision

Today's world is obeserved through the lens of multiple devices. Here, we consider images of our earth took from satellites. All sort of things can be spotted through these images. The idea is to train a quantum model to flag some target components that we could find on these images. Here, for simplicity, we will consider a narrower dataset. 

We assume that our model can only input 10 type of images, all with the same dimensions: planes, airports, ships, harbor, tennis court, basketball court, clouds, dense residential, railway station, river. Our goal is to follow the traffic both in the air and in the water. That means that our model should flag us when it sees any airport, ship, plane, or harbor. 

Knowing an image contains either a ship or a plane is already good, but not sufficient, since these two are broadly different. We can thus train another model that takes in entry an image of either a plane, a boat, a harbor, or an airport, and that classifies the image in its corresponding class. 

To train our models, we used an already existing dataset called `NWPU-RESISC45`. It contains images from satellites classified in 45 classes. We cutted all classes but the previously mentionned classes of interest, and use them as training data and testing data. 

## A sneak peak of our classification model


### The full pipeline and the role of quantum in it
The full pipeline can be understood by thinking of a satellite image of size 32x32 coming as an input to the algorithm. 

The first step is to process the image into 512 characteristic features of the image, using Resnet 18, a pretrained model available. We then use a PCA 10 algorithm to reduce to the 10 most important features. Then, we use these in a QSVM, algorithm, marking the first use of a quantum computer. This QSVM is only trained to say wheter or not an image is flagged as normal, we call it a QSVM 1 class. Thus, we have no idea of which class we obtained. We then need to use another algorithm, which is only applied if the image is flagged as anormal by the QSVM.

The same entry image is thus converted into 10 new features using a classical neural network. These 10 features are used in a photonic circuit that performs a boson sampling method to classify the data. Then, the algorithm as a return value of either `"not_flagged"`, `"airport"`, `"harbor"`, `"ship"`, or `"airplane"`.

Quantum is used as a machine learning engine. All of these task could be done via classical algorithms, thus, the advantage of using quantum is more difficult to see.

<img src="./pipeline.png" alt="Alt Text" width="1000">

### 1. Training a QSVM algorithm to flag or not the image as a targeted image.

The first step of our pipeline is to train our QSVM model to actually recognise the targeted images. Here, we want to recognise if the image is either a plane, a boat, an airport, or a harbor. Then, we apply this trained model on our test data to highlight the potential anomalies. This training is made with 100 random datas, containing 10 datas that should be flagged.

In [1]:

from anomaly_detector import create_and_train_classifier, create_anomaly_dataset, extract_image_features, feature_reductor, create_quantum_kernels, detect_anomaly
from sklearn.metrics import roc_auc_score
import numpy as np

import torch
import random

#Fixing the seed for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)

torch.use_deterministic_algorithms(True)

# ====================================================
# DATA
# ====================================================

train_tensor = create_anomaly_dataset(0.0, 100)

x_train = train_tensor[0][0]
y_train = train_tensor[0][1]

test_tensor = create_anomaly_dataset(0.1, 100)

x_test = test_tensor[1][0]
y_test = test_tensor[1][1]
y_test_classes=test_tensor[1][2]


# ====================================================
# FEATURE EXTRACTION
# ====================================================

train_features = extract_image_features(x_train)
test_features = extract_image_features(x_train)

print("Train features:", train_features.shape)
print("Test features:", test_features.shape)


# ====================================================
# PCA + scaler -> 8 DIMENSIONS
# ====================================================

train_pca = feature_reductor(train_features, num_features=8)
test_pca = feature_reductor(train_features, num_features=8)


# ====================================================
# QUANTUM KERNEL
# ====================================================

train_kernel, test_kernel = create_quantum_kernels(x_train=train_pca, x_test=test_pca, num_features=8)


# ====================================================
# ONE-CLASS SVM
# ====================================================
# Unlike standard SVMs that require data from two classes to find a separating boundary, a One-Class SVM trains only on "normal" data to learn its distribution, flagging anything that deviates significantly from this norm
ocsvm = create_and_train_classifier(train_kernel)


# ====================================================
# ANOMALY SCORES
# ====================================================

decision_scores, preds = detect_anomaly(ocsvm, test_kernel)


# ====================================================
# CONFUSION MATRIX COMPONENTS
# ====================================================

y_test=y_test.numpy()  # Convert to NumPy array if it's a PyTorch tensor

TP = np.sum((preds == 1) & (y_test == 1))
FP = np.sum((preds == 1) & (y_test == 0))
TN = np.sum((preds == 0) & (y_test == 0))
FN = np.sum((preds == 0) & (y_test == 1))

print("\n==============================")
print("CONFUSION MATRIX SUMMARY")
print("==============================")
print(f"True Positives  (TP): {TP}")
print(f"False Positives (FP): {FP}")
print(f"True Negatives  (TN): {TN}")
print(f"False Negatives (FN): {FN}")

Train features: (100, 512)
Test features: (100, 512)
Building training kernel matrix...
Building test kernel matrix...

CONFUSION MATRIX SUMMARY
True Positives  (TP): 2
False Positives (FP): 8
True Negatives  (TN): 82
False Negatives (FN): 8


### 2. Training our classification algorithm
We then want to train the algorithm in charge of classifying the flagged anomalies in order to classify them in their corresponding classes. This model uses 100 data per class to train, meaning we use a total of 400 datas.

In [2]:
from trainer import get_datas, QuantumReservoirNet, train

import torch.nn as nn
import torch
import torch.optim as optim

# nb_classes = 4
anomaly_ratio = 1

x_train, y_train, x_test_classifier, y_test_classifier = get_datas(400, anomaly_ratio)

# Convert to PyTorch tensors
x_train_t = torch.tensor(x_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)

x_test_t = torch.tensor(x_test_classifier, dtype=torch.float32)
y_test_t = torch.tensor(y_test_classifier, dtype=torch.long)

# Define the model, loss function and optimizer
model = QuantumReservoirNet()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

epochs = 100
batch_size = 16

model = train(
    model,
    x_train_t,
    y_train_t,
    x_test_t,
    y_test_t,
    criterion,
    optimizer,
    epochs=epochs,
    batch_size=batch_size,
)

/Users/lfvigneux/miniconda3/envs/compile_live_svp/lib/python3.12/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Epoch 1/100 | Loss: 1.3693 | Train acc: 0.250 | Test acc: 0.250
Epoch 2/100 | Loss: 1.3873 | Train acc: 0.250 | Test acc: 0.250
Epoch 3/100 | Loss: 1.3918 | Train acc: 0.250 | Test acc: 0.250
Epoch 4/100 | Loss: 1.3587 | Train acc: 0.387 | Test acc: 0.370
Epoch 5/100 | Loss: 1.2305 | Train acc: 0.475 | Test acc: 0.477
Epoch 6/100 | Loss: 1.2614 | Train acc: 0.468 | Test acc: 0.463
Epoch 7/100 | Loss: 1.1534 | Train acc: 0.493 | Test acc: 0.480
Epoch 8/100 | Loss: 1.0820 | Train acc: 0.502 | Test acc: 0.480
Epoch 9/100 | Loss: 1.0454 | Train acc: 0.480 | Test acc: 0.452
Epoch 10/100 | Loss: 1.0472 | Train acc: 0.502 | Test acc: 0.480
Epoch 11/100 | Loss: 0.8814 | Train acc: 0.500 | Test acc: 0.450
Epoch 12/100 | Loss: 0.8987 | Train acc: 0.512 | Test acc: 0.485
Epoch 13/100 | Loss: 0.8330 | Train acc: 0.540 | Test acc: 0.485
Epoch 14/100 | Loss: 0.8579 | Train acc: 0.522 | Test acc: 0.493
Epoch 15/100 | Loss: 0.8825 | Train acc: 0.533 | Test acc: 0.475
Epoch 16/100 | Loss: 0.8971 | Trai

### 3. Classify the true positives of step #1
Now, we can use our trained model to try to classify the true positives obtained in the first step. We then try to classify the data we flagged as anomaly and that we know should be anomaly with our classification model.

In [3]:
anomaly_idx = np.where(preds == 1)[0]
class_preds = np.full(len(preds), -1, dtype=int)

if len(anomaly_idx) == 0:
    print("No flagged anomalies in the test set.")
else:
    x_flagged = x_test[anomaly_idx]

    if not isinstance(x_flagged, torch.Tensor):
        x_flagged = torch.tensor(x_flagged, dtype=torch.float32)

    model.eval()
    with torch.no_grad():
        logits_flagged = model(x_flagged)
        flagged_preds = logits_flagged.argmax(dim=1).cpu().numpy()

    class_preds[anomaly_idx] = flagged_preds

    print(f"Classified {len(anomaly_idx)} flagged samples through the second model.")
    for idx, pred in zip(anomaly_idx, flagged_preds):
        print(f"Sample {idx:03d} | predicted class = {pred}")

Classified 10 flagged samples through the second model.
Sample 011 | predicted class = 0
Sample 032 | predicted class = 0
Sample 040 | predicted class = 2
Sample 042 | predicted class = 1
Sample 063 | predicted class = 3
Sample 074 | predicted class = 2
Sample 079 | predicted class = 0
Sample 083 | predicted class = 0
Sample 084 | predicted class = 3
Sample 098 | predicted class = 2


In [7]:
# Ensure numpy
y_test = np.array(y_test)
preds = np.array(preds)
class_preds = np.array(class_preds)
y_class_test = np.array(y_test_classes)

# ====================================================
# 1. TRUE ANOMALIES DETECTED
# ====================================================

true_anomaly_idx = np.where((preds == 1) & (y_test == 1))[0]

print("\n==============================")
print("TRUE ANOMALIES DETECTED")
print("==============================")
print(f"Total detected true anomalies: {len(true_anomaly_idx)}\n")

for i in true_anomaly_idx:
    print(
        f"[TP] Sample {i:03d} | "
        f"Pred Class = {class_preds[i]} | "
        f"True Class = {y_class_test[i]}"
    )

# ======================================================
# 2. MISSED ANOMALIES, compute the expected class values
# ======================================================

missed_anomaly_idx = np.where((preds == 0) & (y_test == 1))[0]

print("\n==============================")
print("MISSED ANOMALIES (FN)")
print("==============================")
print(f"Total missed anomalies: {len(missed_anomaly_idx)}\n")

if len(missed_anomaly_idx) > 0:

    x_missed = x_test[missed_anomaly_idx]

    if not isinstance(x_missed, torch.Tensor):
        x_missed = torch.tensor(x_missed, dtype=torch.float32)

    model.eval()
    with torch.no_grad():
        logits_missed = model(x_missed)
        missed_class_preds = logits_missed.argmax(dim=1).cpu().numpy()

    for idx, pred_class in zip(missed_anomaly_idx, missed_class_preds):

        print(
            f"[FN] Sample {idx:03d} | "
            f"Pred Class = {pred_class} | "
            f"True Class = {y_class_test[idx]}"
        )

# ====================================================
# 3. CLASSIFICATION PERFORMANCE ON TRUE ANOMALIES ONLY
# ====================================================

if len(true_anomaly_idx) > 0:
    acc_on_anomalies = np.mean(
        class_preds[true_anomaly_idx] == y_class_test[true_anomaly_idx]
    )

    print("\n==============================")
    print("CLASSIFICATION ON TRUE ANOMALIES")
    print("==============================")
    print(f"Accuracy on anomalies: {acc_on_anomalies:.4f}")
else:
    print("\nNo true anomalies detected → cannot compute anomaly-class accuracy")


TRUE ANOMALIES DETECTED
Total detected true anomalies: 2

[TP] Sample 032 | Pred Class = 0 | True Class = 1
[TP] Sample 079 | Pred Class = 0 | True Class = 0

MISSED ANOMALIES (FN)
Total missed anomalies: 8

[FN] Sample 005 | Pred Class = 3 | True Class = 3
[FN] Sample 014 | Pred Class = 0 | True Class = 0
[FN] Sample 044 | Pred Class = 2 | True Class = 2
[FN] Sample 055 | Pred Class = 1 | True Class = 1
[FN] Sample 059 | Pred Class = 0 | True Class = 3
[FN] Sample 061 | Pred Class = 0 | True Class = 3
[FN] Sample 070 | Pred Class = 2 | True Class = 2
[FN] Sample 095 | Pred Class = 2 | True Class = 2

CLASSIFICATION ON TRUE ANOMALIES
Accuracy on anomalies: 0.5000


/var/folders/k_/9bm2xqh95599gcr_s25f45740000gn/T/ipykernel_65678/2006476610.py:5: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  y_class_test = np.array(y_test_classes)


# Summary of pipeline

In [8]:
# ====================================================
# ANOMALY DETECTION SUMMARY
# ====================================================

total_samples = len(y_test)

total_anomalies = np.sum(y_test == 1)
total_normals = np.sum(y_test == 0)

detected_anomalies = np.sum((preds == 1) & (y_test == 1))
missed_anomalies = np.sum((preds == 0) & (y_test == 1))

false_positives = np.sum((preds == 1) & (y_test == 0))
true_negatives = np.sum((preds == 0) & (y_test == 0))

anomaly_detection_rate = (
    detected_anomalies / total_anomalies
    if total_anomalies > 0
    else 0.0
)

false_positive_rate = (
    false_positives / total_normals
    if total_normals > 0
    else 0.0
)

print("\n==============================")
print("ANOMALY DETECTION SUMMARY")
print("==============================")
print(f"Total samples:            {total_samples}")
print(f"Total anomalies:          {total_anomalies} ({100 * total_anomalies / total_samples:.2f}%)")
print(f"Total normals:            {total_normals} ({100 * total_normals / total_samples:.2f}%)")
print()
print(f"Detected anomalies (TP):  {detected_anomalies}")
print(f"Missed anomalies (FN):    {missed_anomalies}")
print(f"False positives (FP):     {false_positives}")
print(f"True negatives (TN):      {true_negatives}")
print()
print(f"Detection rate:           {100 * anomaly_detection_rate:.2f}%")
print(f"False positive rate:      {100 * false_positive_rate:.2f}%")


if len(true_anomaly_idx) > 0:

    correct_classes = np.sum(
        class_preds[true_anomaly_idx]
        == y_class_test[true_anomaly_idx]
    )

    total_detected = len(true_anomaly_idx)

    acc_on_anomalies = correct_classes / total_detected

    print("\n==============================")
    print("CLASSIFICATION ON DETECTED ANOMALIES")
    print("==============================")
    print(f"Correctly classified: {correct_classes}")
    print(f"Detected anomalies:   {total_detected}")
    print(f"Classification acc:   {acc_on_anomalies:.4f}")
    print(f"Classification acc:   {100 * acc_on_anomalies:.2f}%")

else:
    print("\nNo true anomalies detected → cannot compute anomaly-class accuracy")


ANOMALY DETECTION SUMMARY
Total samples:            100
Total anomalies:          10 (10.00%)
Total normals:            90 (90.00%)

Detected anomalies (TP):  2
Missed anomalies (FN):    8
False positives (FP):     8
True negatives (TN):      82

Detection rate:           20.00%
False positive rate:      8.89%

CLASSIFICATION ON DETECTED ANOMALIES
Correctly classified: 1
Detected anomalies:   2
Classification acc:   0.5000
Classification acc:   50.00%


## Conclusions
In summary, we did implement two models to classify our data. The first one is used to detect anomaly or not, while the second one uses the anomalies detected and classifies them in the four categories: either an airplane, a boat, a harbor, an airport. 

Results aren't that much good, as we've been able to classify only one anomaly in the whole pipeline, starting with 10 anomalies dispersed in a 100 samples. But the concept is here, and only needs better learning from our model with more computational resources. 

As a final discussion, it is to mention that here, our quantum solution doesn't explicitely offer any advantage over the classical solutions, and that we didn't solve any problem upfront. But maybe quantum, with further work, could expand the possibility of learning for more expressive models, with the ability of adding more parameters in the extended Hilbert space offered by quantum states.